# Hands-on: Find Faulty Sensor Readings with an Autoencoder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/Autoencoder_colab.ipynb)

This notebook lets you **train the sensor autoencoder yourself**, choose the fault and the threshold, try it on
**your own location**, and (extra) use an autoencoder to **clean noisy leaf images**. The ideas behind it
(bottleneck, reconstruction error, threshold) are explained in the presenter's [marimo notebook](https://molab.marimo.io/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/Autoencoder.py).

**No coding needed:** run each cell in order with the ▶ button (or `Shift + Enter`). Cells marked **✏️ Settings**
have a few values you can change before running them. Everything runs on CPU; a GPU speeds up the image part.

## 1. Setup
Downloads the shared workshop code and the hourly weather of Chachapoyas, used as a 'field sensor' (temperature and humidity).

In [ ]:
import os, urllib.request
for name in ["rnn_common.py", "ae_common.py", "cnn_common.py"]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/lobaton/AI-Workshop-UNTRM/main/session2_inside_the_black_box/src/{name}", name)

import numpy as np
import matplotlib.pyplot as plt
import torch
import rnn_common as rc
import ae_common as ac

plt.rcParams.update({"font.size": 14, "lines.linewidth": 3})
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", "GPU" if device == "cuda" else "CPU")

weather = rc.load_weather(rc.fetch(rc.WEATHER_FILE))
print("Fault types:")
for kind, text in ac.FAULTS.items():
    print(f"  {kind}: {text}")

## 2. ✏️ Settings

- `LATENT`: bottleneck size, how many numbers summarize one day (1 to 48).
- `EPOCHS`: passes through the training data.
- `FAULT_KIND`: one of `"Stuck sensor"`, `"Spike"`, `"Drift"`, `"Dropout to zero"`.
- `FAULT_SIZE`: 1 (subtle) to 5 (obvious).
- `THRESHOLD_PERCENTILE`: a day is flagged if its error is above this percentile of the training errors.

In [ ]:
LATENT = 4
EPOCHS = 20
FAULT_KIND = "Stuck sensor"
FAULT_SIZE = 2
THRESHOLD_PERCENTILE = 99.0

## 3. Train and detect

The autoencoder learns from the training period (assumed normal). Then the chosen fault is added to half of
the test days, and every test day whose reconstruction error is above the threshold is flagged.

In [ ]:
def train_and_detect(weather, label):
    days = ac.SensorDays(weather)
    x_train, x_val = days.scale(days.sliding("train")), days.scale(days.sliding("validation", stride=24))
    torch.manual_seed(0)
    model = ac.SensorAE(LATENT)
    history = ac.train_ae(model, x_train, x_val, epochs=EPOCHS)
    train_err = ac.reconstruction_error(model, x_train)
    threshold = np.percentile(train_err, THRESHOLD_PERCENTILE)

    test_days, _ = days.calendar_days("test")
    faulty, labels = ac.faulty_test_set(test_days, FAULT_KIND, FAULT_SIZE)
    test_err = ac.reconstruction_error(model, days.scale(faulty))
    flags = test_err > threshold
    found, false = int((flags & (labels == 1)).sum()), int((flags & (labels == 0)).sum())
    print(f"{label}: bottleneck {LATENT}, trained in {history['seconds']:.0f} s")
    print(f"  Faults found: {found} of {labels.sum()}   False alarms: {false} of {(labels == 0).sum()} normal days")

    bins = np.logspace(np.log10(min(train_err.min(), test_err.min()) + 1e-6),
                       np.log10(max(train_err.max(), test_err.max())), 40)
    plt.figure(figsize=(11, 4))
    for values, color, name in [(train_err, "#BBBBBB", "Training (normal)"),
                                (test_err[labels == 0], "#0072B2", "Test: normal"),
                                (test_err[labels == 1], "#D55E00", "Test: with fault")]:
        plt.hist(values, bins=bins, color=color, alpha=0.7, label=name,
                 weights=np.full(len(values), 100 / len(values)))
    plt.axvline(threshold, color="black", linestyle="--", label="Threshold")
    plt.xscale("log"); plt.xlabel("Reconstruction error"); plt.ylabel("Share of group (%)")
    plt.title(label); plt.legend(fontsize=11); plt.show()
    return model, days, faulty, labels

model, days, faulty, labels = train_and_detect(weather, "Chachapoyas")

**Try:**
- `LATENT = 1` or `2` vs. `48`: which faults are missed, and when does the network start copying the fault?
- `FAULT_SIZE = 1` for each fault type: which faults are hardest to detect?
- `THRESHOLD_PERCENTILE = 95` vs. `99.9`: how do found faults and false alarms change?

## 4. Look at one faulty day
The input day (with the fault) and what the autoencoder rebuilt.

In [ ]:
i = int(np.flatnonzero(labels)[0])
rebuilt = days.unscale(ac.reconstruct(model, days.scale(faulty[i:i + 1])))[0]
plt.figure(figsize=(10, 4))
plt.plot(faulty[i, :, 0], "o-", color="gray", label="Input (with fault)")
plt.plot(rebuilt[:, 0], color="#D55E00", label="Reconstruction")
plt.xlabel("Hour of day"); plt.ylabel("Temperature (°C)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. ✏️ Your own location

Change `LAT` and `LON` to any place and run the next two cells. The weather is downloaded live from NASA POWER
(about 1 minute) and the whole experiment is repeated with the settings from Section 2.
*South latitudes and west longitudes are negative.*

In [ ]:
LAT, LON = -5.74, -78.44   # Example: Bagua Grande, Amazonas
PLACE = "Bagua Grande"

In [ ]:
my_weather = rc.fetch_nasa_power(LAT, LON, "20230101", "20241231")
_ = train_and_detect(my_weather, PLACE)

## 6. Extra: cleaning noisy leaf images (denoising autoencoder)

An autoencoder can also be trained to **remove noise**: we give it a noisy image as input and ask it to output
the clean image. This is the idea behind denoising in microscopy and low-light photography. We use the bean-leaf
photos from the CNN notebook, reduced to 64×64 pixels.

✏️ `NOISE_LEVEL` controls how much noise is added (0.1 = light, 0.4 = heavy).

In [ ]:
NOISE_LEVEL = 0.2
IMAGE_EPOCHS = 15

In [ ]:
import cnn_common as cc
from torch import nn

beans = cc.load_beans(cc.fetch(cc.BEANS_FILE))
to_small = lambda imgs: nn.functional.avg_pool2d(
    torch.from_numpy(np.array(imgs)).permute(0, 3, 1, 2).float() / 255, 2)  # 128 -> 64 pixels, values 0-1
clean_train, clean_test = to_small(beans["train"][0]), to_small(beans["test"][0])

class DenoisingAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),
                                     nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU())   # 64 -> 16 pixels
        self.decoder = nn.Sequential(nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
                                     nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1), nn.Sigmoid())
    def forward(self, x):
        return self.decoder(self.encoder(x))

torch.manual_seed(0)
denoiser = DenoisingAE().to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=2e-3)
for epoch in range(IMAGE_EPOCHS):
    for idx in torch.randperm(len(clean_train)).split(32):
        clean = clean_train[idx].to(device)
        noisy = (clean + NOISE_LEVEL * torch.randn_like(clean)).clamp(0, 1)
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(denoiser(noisy), clean)   # target = the CLEAN image
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}/{IMAGE_EPOCHS}: loss {loss.item():.4f}")

In [ ]:
torch.manual_seed(1)
clean = clean_test[:4]
noisy = (clean + NOISE_LEVEL * torch.randn_like(clean)).clamp(0, 1)
with torch.no_grad():
    cleaned = denoiser(noisy.to(device)).cpu()

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for j in range(4):
    for row, (img, name) in enumerate([(noisy, "Noisy input"), (cleaned, "Denoised"), (clean, "Original")]):
        axes[row, j].imshow(img[j].permute(1, 2, 0).numpy())
        axes[row, j].set_xticks([]); axes[row, j].set_yticks([])
        if j == 0:
            axes[row, j].set_ylabel(name, fontsize=14)
plt.tight_layout(); plt.show()

## Questions for discussion

1. In your own work (sensors, field data, microscopy), what would count as an "anomaly"? Do you have enough
   **normal** data to train an autoencoder?
2. The denoised leaves look smoother but slightly blurry. When would that blur be a problem, for example when
   counting spores or measuring lesions?